# Question 4

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_curve,
    roc_auc_score
)


In [2]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

# Load dataset
spambase = fetch_ucirepo(id=94)
X = spambase.data.features
y = spambase.data.targets.iloc[:, 0]   # make y 1D

# Same split used in Problem 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [3]:
def k_fold_cv_validation_error(model, X, y, k):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_errors = []

    for train_index, val_index in skf.split(X, y):
        X_train_fold = X.iloc[train_index]
        X_val_fold = X.iloc[val_index]
        y_train_fold = y.iloc[train_index]
        y_val_fold = y.iloc[val_index]

        model.fit(X_train_fold, y_train_fold)
        y_val_pred = model.predict(X_val_fold)

        val_accuracy = accuracy_score(y_val_fold, y_val_pred)
        val_error = 1 - val_accuracy
        fold_errors.append(val_error)

    avg_val_error = np.mean(fold_errors)
    return fold_errors, avg_val_error

In [4]:
# models

logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter = 5000))
])

lda_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis())
])

k_values = [5, 10]
results = []

for k in k_values:
    logreg_fold_errors, logreg_avg_error = k_fold_cv_validation_error(logreg_model, X, y, k)
    lda_fold_errors, lda_avg_error = k_fold_cv_validation_error(lda_model, X, y, k)

    print(f"\nLogistic Regression with k = {k}")
    print("Fold validation errors:", logreg_fold_errors)
    print("Average validation error:", logreg_avg_error)

    print(f"\nLDA with k = {k}")
    print("Fold validation errors:", lda_fold_errors)
    print("Average validation error:", lda_avg_error)

    results.append({
        "model": "Logistic Regression",
        "k": k,
        "average_validation_error": logreg_avg_error
    })

    results.append({
        "model": "LDA",
        "k": k,
        "average_validation_error": lda_avg_error
    })

results_df = pd.DataFrame(results)
print("\nSummary of average validation errors:")
print(results_df)


Logistic Regression with k = 5
Fold validation errors: [0.08686210640608039, 0.06630434782608696, 0.07173913043478264, 0.07608695652173914, 0.07608695652173914]
Average validation error: 0.07541589954208565

LDA with k = 5
Fold validation errors: [0.12160694896851254, 0.10652173913043483, 0.09673913043478266, 0.12173913043478257, 0.125]
Average validation error: 0.11432138979370252

Logistic Regression with k = 10
Fold validation errors: [0.08459869848156187, 0.07826086956521738, 0.07173913043478264, 0.06739130434782614, 0.09130434782608698, 0.06304347826086953, 0.06521739130434778, 0.07391304347826089, 0.060869565217391286, 0.08913043478260874]
Average validation error: 0.07454682636989532

LDA with k = 10
Fold validation errors: [0.1171366594360087, 0.12608695652173918, 0.10434782608695647, 0.10217391304347823, 0.09999999999999998, 0.10217391304347823, 0.11956521739130432, 0.11086956521739133, 0.12173913043478257, 0.12391304347826082]
Average validation error: 0.11280062246533999

S

Logistic regression performs better than LDA for both values of k, because it has the lower average validation error for both. The average validation errors were:

*   logsitic regression, k= 5: 0.0754
*   LDA, k = 5: 0.1143
* logistic regression, k=10: 0.0745
* LDA, k = 10: 0.1128

So, logistic regression performs better for both k=5 and k=10. These results are also consistent with problem 3, which suggests that logistic regression is the more reliable classifier on the SPAMBASE dataset.



